# Feature Store Streaming Demo - Train & Deploy

Trains a next-item recommendation model (XGBoost ranker) using:
- USER_PROFILE_FV (batch) - demographics
- ITEM_PROFILE_FV (batch) - catalog attributes
- USER_EVENTS_BATCH_FV (batch) - behavioral event data

Deploys to the Snowflake Model Registry for online inference.

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.41" snowflake-connector-python requests

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, CreationMode

session = get_active_session()
session.use_schema("ML_DEMOS.ONLINE_W_STREAMING")
session.use_warehouse("ML_DEMO_WH")

fs = FeatureStore(
    session=session, database="ML_DEMOS", name="ONLINE_W_STREAMING",
    default_warehouse="ML_DEMO_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Connected.")

In [ ]:
# Retrieve registered feature views
registered_user_profile_fv = fs.get_feature_view("USER_PROFILE_FV", "V1")
registered_item_profile_fv = fs.get_feature_view("ITEM_PROFILE_FV", "V1")
registered_user_events_batch_fv = fs.get_feature_view("USER_EVENTS_BATCH_FV", "V1")

print("Feature views loaded:")
print(f"  {registered_user_profile_fv.name}")
print(f"  {registered_item_profile_fv.name}")
print(f"  {registered_user_events_batch_fv.name}")

In [ ]:
# Build training spine from RAW_EVENTS
# Each row = (USER_ID, ITEM_ID, EVENT_TS, RELEVANCE_LABEL)
# Label encodes interaction strength for ranking

spine_df = session.sql('''
    SELECT
        USER_ID,
        ITEM_ID,
        EVENT_TS,
        CASE EVENT_TYPE
            WHEN 'purchase' THEN 3
            WHEN 'add_to_cart' THEN 2
            WHEN 'click' THEN 1
            ELSE 0
        END AS RELEVANCE_LABEL
    FROM ML_DEMOS.ONLINE_W_STREAMING.RAW_EVENTS
    WHERE ITEM_ID IS NOT NULL
''')

print(f"Spine rows: {spine_df.count()}")
spine_df.show(10)

In [ ]:
# Generate training set with point-in-time correct feature joins
training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[
        registered_user_events_batch_fv,   # behavioral events (offline)
        registered_user_profile_fv,         # demographics (batch)
        registered_item_profile_fv,         # item catalog (batch)
    ],
    spine_timestamp_col="EVENT_TS",
    spine_label_cols=["RELEVANCE_LABEL"],
)

print(f"Training set columns: {training_df.columns}")
print(f"Training set rows: {training_df.count()}")
training_df.show(5)

In [ ]:
# Convert to pandas for model training
train_pd = training_df.to_pandas()
print(f"Shape: {train_pd.shape}")
print(f"\nLabel distribution:")
print(train_pd["RELEVANCE_LABEL"].value_counts().sort_index())
print(f"\nNull counts:")
print(train_pd.isnull().sum()[train_pd.isnull().sum() > 0])

In [ ]:
# Prepare features for XGBoost
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

NUMERIC_FEATURES = [
    "PRICE", "AVG_RATING",
    "VIEW_COUNT_1H", "VIEW_COUNT_24H", "CART_COUNT_1H",
    "PURCHASE_COUNT_24H", "TOTAL_DWELL_SEC_1H", "UNIQUE_ITEMS_VIEWED_1H",
]
CATEGORICAL_FEATURES = ["COUNTRY", "USER_SEGMENT", "DEVICE_TYPE", "AGE_GROUP", "CATEGORY", "SUBCATEGORY", "BRAND"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

numeric_transformer = SimpleImputer(strategy="constant", fill_value=0)
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

X = train_pd[FEATURE_COLUMNS]
y = train_pd["RELEVANCE_LABEL"]

print(f"Features: {FEATURE_COLUMNS}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Train XGBoost ranker via pipeline
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        objective="multi:softprob", num_class=4, random_state=42,
        enable_categorical=False,
    )),
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["view", "click", "add_to_cart", "purchase"]))

In [ ]:
# Feature importance
import matplotlib.pyplot as plt

importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)
importance.plot(kind="barh", figsize=(8, 5), title="Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Log model to Snowflake Model Registry
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name="ML_DEMOS", schema_name="ONLINE_W_STREAMING")

mv = registry.log_model(
    pipeline,
    model_name="RECOMMENDATION_RANKER",
    version_name="V1",
    sample_input_data=X_train.head(5),
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    comment="XGBoost next-item recommendation ranker with preprocessing pipeline",
)
print(f"Model logged: {mv.model_name} / {mv.version_name}")

In [ ]:
# Drop existing service and redeploy with V2
session.sql("DROP SERVICE IF EXISTS ML_DEMOS.ONLINE_W_STREAMING.RECOMMENDATION_RANKER_ONLINE_SERVICE").collect()

mv = registry.get_model("RECOMMENDATION_RANKER").version("V2")

mv.create_service(
    service_name="RECOMMENDATION_RANKER_ONLINE_SERVICE",
    service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    gpu_requests=None,
    autocapture=True
)
print("Check status with: SHOW SERVICES IN SCHEMA ML_DEMOS.ONLINE_W_STREAMING;")